In [1]:
!pip install transformers==4.44.2 joblib==1.4.2 scikit-learn==1.6.0 numpy==1.26.4 pandas==2.2.3 scipy==1.13.1 seaborn==0.13.2 tqdm==4.66.5 lightgbm==4.5.0 xgboost==2.1.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 72.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 105.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 48.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 MB 11.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.7 MB/s eta 0:00:00:00:01
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Un

In [4]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [5]:
# Load datasets
train_df = pd.read_csv('/kaggle/input/mdck-dataset/Train_MDCK.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/mdck-dataset/Test_MDCK.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [6]:
tokenizer = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
model = AutoModelForSequenceClassification.from_pretrained('seyonec/ChemBERTa-zinc-base-v1', num_labels=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

#dataset class
class SMILESDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=325):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        smiles = self.dataframe.iloc[idx]['SMILES']
        permeability = self.dataframe.iloc[idx]['Permeability']
        inputs = self.tokenizer(smiles, return_tensors='pt', padding="max_length", truncation=True, max_length=self.max_length)
        
        input_ids = inputs['input_ids'].squeeze(0)  # Shape: (sequence_length,)
        attention_mask = inputs['attention_mask'].squeeze(0)  # Shape: (sequence_length,)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(permeability, dtype=torch.float)
        }


# datasets
train_dataset = SMILESDataset(train_df, tokenizer)
test_dataset = SMILESDataset(test_df, tokenizer)
batch_size = 16
# data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at seyonec/ChemBERTa-zinc-base-v1 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 20

In [8]:
# Training loop
from tqdm import tqdm
for epoch in range(num_epochs):
    print(f"Entered Epoch {epoch + 1}")
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f'Training Epoch {epoch + 1}/{num_epochs}', unit='batch'):
        optimizer.zero_grad()

        # Move all batch tensors to device
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"].unsqueeze(1)  # still shape: (batch_size, 1)

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=labels
        )
        loss = outputs.loss
        train_loss += loss.item()

        # Backprop and optimizer step
        loss.backward()
        optimizer.step()

    avg_train_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}')

Entered Epoch 1


Training Epoch 1/20: 100%|██████████| 4/4 [00:02<00:00,  1.48batch/s]


Epoch 1/20 - Train Loss: 18.2276
Entered Epoch 2


Training Epoch 2/20: 100%|██████████| 4/4 [00:01<00:00,  3.24batch/s]


Epoch 2/20 - Train Loss: 3.3808
Entered Epoch 3


...

Epoch 18/20 - Train Loss: 0.4638
Entered Epoch 19


Training Epoch 19/20: 100%|██████████| 4/4 [00:01<00:00,  3.04batch/s]


Epoch 19/20 - Train Loss: 0.4299
Entered Epoch 20


Training Epoch 20/20: 100%|██████████| 4/4 [00:01<00:00,  3.04batch/s]

Epoch 20/20 - Train Loss: 0.4858


In [9]:
model_name = 'ChemBERTa_model_1_mdck'
model_save_path = f'/kaggle/working/{model_name}'
os.makedirs(model_save_path, exist_ok=True)

tokenizer.save_pretrained(model_save_path)
model.save_pretrained(model_save_path)

print(f'Model and tokenizer saved to {model_save_path}')

Model and tokenizer saved to /kaggle/working/ChemBERTa_model_1_mdck


In [10]:
from scipy.stats import pearsonr, spearmanr

model.eval()
test_loss = 0
test_true_labels = []
predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing', unit='batch'):
      
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].unsqueeze(1).to(device).float()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        test_loss += loss.item()

        test_true_labels.extend(labels.cpu().numpy())
        preds = outputs.logits.squeeze().cpu().numpy()  
        predictions.extend(preds)

# Final test loss
avg_test_loss = test_loss / len(test_loader)
print(f'Test Loss: {avg_test_loss:.4f}')

test_true_labels = np.array(test_true_labels).flatten()
predictions = np.array(predictions)
print(test_true_labels.shape)
print(predictions.shape)

# Performance metrics
mse = mean_squared_error(test_true_labels, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_true_labels, predictions)
r2 = r2_score(test_true_labels, predictions)
PCC,_ = pearsonr(test_true_labels, predictions)
SCC,_ = spearmanr(test_true_labels, predictions)
# Print performance metrics
print(f'Mean Squared Error: {mse:.4f}')
print(f'Root Mean Squared Error: {rmse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'R^2 Score: {r2:.4f}')
print(f'Pearson Correlation Coefficient: {PCC:.4f}')
print(f'Spearman Correlation Coefficient: {SCC:.4f}')

# Print hyperparameters
print("Hyperparameters:")
print(f"Learning Rate: {5e-5}")
print(f"Batch Size: 16")
print(f"Epochs: {num_epochs}")

Testing: 100%|██████████| 1/1 [00:00<00:00,  5.32batch/s]


Test Loss: 0.4466
(13,)
(13,)
Mean Squared Error: 0.4466
Root Mean Squared Error: 0.6682
Mean Absolute Error: 0.5749
R^2 Score: 0.3756
Pearson Correlation Coefficient: 0.6747
Spearman Correlation Coefficient: 0.4683
Hyperparameters:
Learning Rate: 5e-05
Batch Size: 16
Epochs: 20


In [11]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model_name = 'ChemBERTa_model_1_mdck'
model_save_path = f'/kaggle/working/{model_name}'

if not os.path.exists(model_save_path):
    raise FileNotFoundError(f"The model directory {model_save_path} does not exist.")

tokenizer = AutoTokenizer.from_pretrained(model_save_path)
model = AutoModel.from_pretrained(model_save_path).to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at /kaggle/working/ChemBERTa_model_1_mdck and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# Load your datasets
train_df = pd.read_csv('/kaggle/input/mdck-dataset/Train_MDCK.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/mdck-dataset/Test_MDCK.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [13]:
train_encodings = tokenizer(list(train_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")
test_encodings = tokenizer(list(test_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")

In [14]:
from tqdm import tqdm 
batch_size = 16 

def generate_embeddings(encodings, batch_size):
    embeddings = []
    model.eval() 
    with torch.no_grad():
        for i in tqdm(range(0, len(encodings['input_ids']), batch_size), desc="Processing batches"):
            batch = {key: val[i:i + batch_size].to(device) for key, val in encodings.items()}  
            outputs = model(**batch)
            embeddings.append(outputs.last_hidden_state)
    return torch.cat(embeddings, dim=0)

In [15]:
train_embeddings = generate_embeddings(train_encodings, batch_size)
print(train_embeddings.shape)
train_embeddings = torch.mean(train_embeddings, dim=1)
print(train_embeddings.shape)

Processing batches: 100%|██████████| 4/4 [00:00<00:00, 16.77it/s]

torch.Size([51, 151, 768])
torch.Size([51, 768])


In [16]:
column_names = [f'x_fine_emb_ChemBerta{i}' for i in range(train_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=train_embeddings.cpu().numpy(), columns=column_names)
train_data = pd.concat([train_df, embeddings_df], axis=1)

In [17]:
test_embeddings = generate_embeddings(test_encodings, batch_size)
print(test_embeddings.shape)
test_embeddings = torch.mean(test_embeddings, dim=1)
print(test_embeddings.shape)

Processing batches: 100%|██████████| 1/1 [00:00<00:00, 138.34it/s]

torch.Size([13, 157, 768])
torch.Size([13, 768])


In [18]:
column_names = [f'x_fine_emb_ChemBerta{i}' for i in range(test_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=test_embeddings.cpu().numpy(), columns=column_names)
test_data = pd.concat([test_df, embeddings_df], axis=1)

In [19]:
train_data.to_csv("/kaggle/working/Train_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_mdck.csv",index=False)
test_data.to_csv("/kaggle/working/Test_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_mdck.csv",index=False)

In [20]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [21]:
train_data = pd.read_csv("/kaggle/working/Train_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_mdck.csv")
test_data = pd.read_csv("/kaggle/working/Test_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_mdck.csv")

In [22]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -4.0)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [23]:
X_train = train_data.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_data['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

X_test = test_data.drop(['ID','SMILES','Permeability'],axis=1)
y_test = test_data['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 768)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 768)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001633 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2355
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 157
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4617,0.5419,0.6795,0.1225,0.3996,0.3598,0.6311,0.6407,0.7944,0.1176,0.3516,0.3388
DecisionTreeRegressor,0.7352,0.7162,0.8574,-0.3973,0.2827,0.1595,0.6158,0.6151,0.7847,0.1389,0.4173,0.4303
RandomForestRegressor,0.4259,0.5241,0.6526,0.1905,0.4482,0.3933,0.4874,0.5913,0.6981,0.3184,0.6181,0.4821
GradientBoostingRegressor,0.4601,0.5440,0.6783,0.1256,0.4343,0.3472,0.5446,0.5851,0.7380,0.2385,0.4968,0.4380
AdaBoostRegressor,0.5027,0.5771,0.7090,0.0445,0.3730,0.3475,0.5039,0.5781,0.7098,0.2954,0.5572,0.4766
XGBRegressor,0.4661,0.5612,0.6827,0.1143,0.4722,0.3720,0.5106,0.5941,0.7146,0.2860,0.5393,0.3857
ExtraTreesRegressor,0.4699,0.5633,0.6855,0.1069,0.4106,0.3301,0.4650,0.5524,0.6819,0.3498,0.6107,0.5069
LinearRegression,1.2736,0.9210,1.1285,-1.4205,0.4067,0.3533,1.3688,1.0251,1.1699,-0.9140,0.1525,0.1625
KNeighborsRegressor,0.5148,0.5928,0.7175,0.0215,0.3786,0.3438,0.4724,0.5607,0.6873,0.3394,0.6258,0.5090
SVR,0.4534,0.5531,0.6733,0.1383,0.3877,0.3785,0.5190,0.5862,0.7205,0.2742,0.5502,0.4904


In [24]:
result_df

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4617,0.5419,0.6795,0.1225,0.3996,0.3598,0.6311,0.6407,0.7944,0.1176,0.3516,0.3388
DecisionTreeRegressor,0.7352,0.7162,0.8574,-0.3973,0.2827,0.1595,0.6158,0.6151,0.7847,0.1389,0.4173,0.4303
RandomForestRegressor,0.4259,0.5241,0.6526,0.1905,0.4482,0.3933,0.4874,0.5913,0.6981,0.3184,0.6181,0.4821
GradientBoostingRegressor,0.4601,0.5440,0.6783,0.1256,0.4343,0.3472,0.5446,0.5851,0.7380,0.2385,0.4968,0.4380
AdaBoostRegressor,0.5027,0.5771,0.7090,0.0445,0.3730,0.3475,0.5039,0.5781,0.7098,0.2954,0.5572,0.4766
XGBRegressor,0.4661,0.5612,0.6827,0.1143,0.4722,0.3720,0.5106,0.5941,0.7146,0.2860,0.5393,0.3857
ExtraTreesRegressor,0.4699,0.5633,0.6855,0.1069,0.4106,0.3301,0.4650,0.5524,0.6819,0.3498,0.6107,0.5069
LinearRegression,1.2736,0.9210,1.1285,-1.4205,0.4067,0.3533,1.3688,1.0251,1.1699,-0.9140,0.1525,0.1625
KNeighborsRegressor,0.5148,0.5928,0.7175,0.0215,0.3786,0.3438,0.4724,0.5607,0.6873,0.3394,0.6258,0.5090
SVR,0.4534,0.5531,0.6733,0.1383,0.3877,0.3785,0.5190,0.5862,0.7205,0.2742,0.5502,0.4904


In [25]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.280759402592662, -5.249559575720341, -6.03...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.255200261137721, -5.412164908318004, -5.0...","[-5.212019349876455, -5.325856461984907, -5.17...","[0.11787421098215432, 0.1955384285274057, 0.10..."
1,DecisionTreeRegressor,"[-4.58, -4.58, -6.3, -5.65, -7.698970004, -6.0...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -4.58, -5.129011186, -6.22, -5.744727...","[-5.0961056402, -5.3539508294, -5.544420234599...","[0.614437672035247, 0.6574905744107005, 0.3141..."
2,RandomForestRegressor,"[-5.473216825030004, -5.171665490749997, -6.05...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.164131188630002, -5.600344831650002, -5.1...","[-5.323796940122001, -5.581150013518003, -5.31...","[0.19195775745536187, 0.1259432126752626, 0.11..."
3,GradientBoostingRegressor,"[-5.134726122822216, -4.929250275365072, -6.21...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.059165672774972, -5.524777281017131, -5.1...","[-5.045435516477403, -5.479896351066589, -5.40...","[0.17125718940219017, 0.18822484934866776, 0.1..."
4,AdaBoostRegressor,"[-5.287624293933334, -5.22, -6.05, -5.45800988...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.7075, -5.785376696, -5.22, -5.45800988428...","[-5.110963017406666, -5.692103318074244, -5.33...","[0.3099652510369491, 0.28722857227322596, 0.16..."
5,XGBRegressor,"[-5.1244245, -4.9367876, -6.249383, -5.694308,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.717027, -6.012428, -4.9965696, -5.778454,...","[-5.0016007, -5.6299124, -5.257386, -5.5726104...","[0.2431203, 0.42990503, 0.20179436, 0.12482180..."
6,ExtraTreesRegressor,"[-5.427731074160001, -4.898184965109996, -6.17...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.046818951230004, -5.760860759220003, -5.2...","[-5.150268423006002, -5.632852246794003, -5.37...","[0.28799766444860975, 0.19856732528426704, 0.1..."
7,LinearRegression,"[-6.890628354717775, -4.138471938361276, -6.80...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.376530915789804, -7.214781482770679, -4.0...","[-4.230855006094097, -6.616032052760026, -4.85...","[0.3099003934403488, 0.9951281784805518, 1.074..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.3999999999999995, -6.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.266666666666667, -5.503333333333333, -5.4...","[-5.436, -5.440666666666667, -5.4925404258, -5...","[0.20588454153832048, 0.18226232621020544, 0.0..."
9,SVR,"[-5.38806389853763, -5.347155694752704, -5.974...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.097828839013021, -5.460290226713058, -5.2...","[-5.248625760386652, -5.448844482104704, -5.32...","[0.27333313611525467, 0.1971117288629571, 0.05..."


In [26]:
result_df.to_csv('/kaggle/working/Results_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_mdck.csv')
prediction_df.to_csv('/kaggle/working/Prediction_data_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_mdck.csv')